# SOXX 基准相对架构与成熟反馈专家

假设、候选和成本在执行前固定于 [plan.json](plan.json)。本轮一个在线候选与固定混合对照，原生 Qlib 数据、Dataset、Recorder、回放及分析构成证据链。历史均已复用，不能取得生产资格。

设计见 [architecture.md](architecture.md)，8项外部一手资料见 [research_sources.md](research_sources.md)。

In [1]:
from pathlib import Path
import sys, json, os
# Qlib uses the existing local MLflow Recorder store; explicit supported opt-in.
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CARD = ROOT / "research" / "benchmark_relative_architecture_20261009"
PLAN = json.loads((CARD / "plan.json").read_text())
import qlib
from qlib.data import D
from qlib.workflow import R
qlib.init(provider_uri=str(ROOT / PLAN["provider_uri"]), region="us", kernels=1)
calendar = D.calendar(freq="day", future=False)
print("Native Qlib calendar:", calendar[0], calendar[-1], "sessions:", len(calendar))
print("Evidence:", PLAN["evidence_kind"], "candidate budget:", PLAN["candidate_budget"])

Native Qlib calendar: 2010-03-11 00:00:00 2026-09-25 00:00:00 sessions: 4162
Evidence: reused_history candidate budget: 1


## 已登记的因果研究路径

数据层以 D.features 读取实际开盘价，再经 DatasetH.prepare 读回。模型只使用 `return_available_at < decision_at` 的净收益；当前和未来收益不进入当前目标。每个成本情景独立运行3个专家、主候选及固定混合，共10条完整原生回放。

In [2]:
from src.workflow.benchmark_relative_research import run_benchmark_relative_research
summary, tables, conclusion = run_benchmark_relative_research(
    source_root=ROOT, evidence_root=ROOT, card_dir=CARD,
    recorder_uri=CARD / "qlib_recorder",
)
print({k: summary[k] for k in ["recorder_id", "native_paths", "native_all_pass", "historical_gate_pass", "promotion_eligible"]})

backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

Native expert passed: soxx, 10bp


backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

Native expert passed: leveraged, 10bp


backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

...

backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

Native portfolio passed: online_fixed_share, 25bp


backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

Native portfolio passed: fixed_mixture, 25bp


{'recorder_id': '64c03a9f9c824e058a2bb360eb6aba11', 'native_paths': 10, 'native_all_pass': True, 'historical_gate_pass': False, 'promotion_eligible': False}


In [3]:
from IPython.display import display, Markdown
display(tables["backtest_benchmark_comparison.csv"][["strategy", "cost_bps", "window", "return_pct", "relative_wealth", "max_drawdown_pct", "cvar5_pct", "turnover"]])
display(tables["gates.csv"])
display(tables["rolling_summary.csv"])
display(tables["ablation_comparison.csv"][["cost_bps", "window", "relative_wealth", "annualized_log_excess"]])

,strategy,cost_bps,window,return_pct,relative_wealth,max_drawdown_pct,cvar5_pct,turnover
0,soxx,10.0,development,325.747970,1.000000,-47.373290,-4.416437,0.999001
1,soxx,10.0,reused_diagnostic,398.621662,1.000000,-42.789189,-5.295891,0.000000
2,leveraged,10.0,development,451.130871,1.294500,-61.328884,-6.139410,7.949436
3,leveraged,10.0,reused_diagnostic,612.213924,1.428365,-56.841908,-7.334959,4.508201
4,defensive,10.0,development,233.039598,0.782246,-39.413111,-3.531787,2.528523
5,defensive,10.0,reused_diagnostic,273.053041,0.748169,-35.458097,-4.236716,1.103415
6,online_fixed_share,10.0,development,331.368643,1.013202,-48.754365,-4.565979,1.960878
7,online_fixed_share,10.0,reused_diagnostic,408.255941,1.019322,-44.287303,-5.493366,0.667566
8,fixed_mixture,10.0,development,330.800555,1.011868,-48.688858,-4.553155,2.017164
9,fixed_mixture,10.0,reused_diagnostic,407.695670,1.018198,-44.197323,-5.458967,0.691956


,cost_bps,aggregate_positive_net_log_excess,minimum_positive_signal_years,median_annual_log_excess_positive,leave_any_one_year_out_log_excess_positive,anchored_max_drawdown_not_worse_than_soxx,cvar5_not_worse_than_soxx,minimum_rolling_positive_excess_rate,stationary_block_20_lower_bound_positive,stationary_block_60_lower_bound_positive,all_pass,positive_years,years,minimum_positive_years
0,10.0,True,False,True,True,False,False,False,False,False,False,5,7,6
1,25.0,True,False,True,False,False,False,False,False,False,False,5,7,6


,cost_bps,rolling_sessions,rolling_windows,positive_windows,positive_excess_rate,minimum_positive_excess_rate,minimum_log_excess,median_log_excess,maximum_log_excess,overlapping_windows,independent_observations
0,10.0,252,1511,886,0.586367,0.6,-0.025271,0.004599,0.023972,True,False
1,25.0,252,1511,876,0.579749,0.6,-0.025512,0.004322,0.023676,True,False


,cost_bps,window,relative_wealth,annualized_log_excess
0,10.0,development,1.001319,0.000188
1,10.0,reused_diagnostic,1.001104,0.000298
2,25.0,development,1.001368,0.000195
3,25.0,reused_diagnostic,1.001057,0.000285


## 中文结论

以下内容由本次 Recorder 结果生成，包含运行范围、指标、年度/窗口差异、晋升决定、风险及下一行动。

In [4]:
display(Markdown(conclusion))

# 基准相对架构与在线专家实验结论

运行了一个固定分享在线候选、固定混合对照及三个静态专家；全部独立扣费结算并通过原生 Qlib 回放。没有训练新预测网络或搜索参数。

每侧25bp，以下按信号年度切片，持仓与学习状态跨年连续：

- development / soxx：收益325.11%，相对SOXX财富1.0000，MaxDD -47.37%，CVaR5 -4.42%，换手1.00。
- reused_diagnostic / soxx：收益398.62%，相对SOXX财富1.0000，MaxDD -42.79%，CVaR5 -5.30%，换手0.00。
- development / leveraged：收益444.60%，相对SOXX财富1.2811，MaxDD -61.40%，CVaR5 -6.14%，换手7.95。
- reused_diagnostic / leveraged：收益607.41%，相对SOXX财富1.4187，MaxDD -56.90%，CVaR5 -7.34%，换手4.51。
- development / defensive：收益231.78%，相对SOXX财富0.7805，MaxDD -39.44%，CVaR5 -3.53%，换手2.53。
- reused_diagnostic / defensive：收益272.44%，相对SOXX财富0.7469，MaxDD -35.48%，CVaR5 -4.24%，换手1.10。
- development / online_fixed_share：收益330.09%，相对SOXX财富1.0117，MaxDD -48.76%，CVaR5 -4.57%，换手1.96。
- reused_diagnostic / online_fixed_share：收益407.71%，相对SOXX财富1.0182，MaxDD -44.30%，CVaR5 -5.49%，换手0.67。
- development / fixed_mixture：收益329.50%，相对SOXX财富1.0103，MaxDD -48.70%，CVaR5 -4.55%，换手2.02。
- reused_diagnostic / fixed_mixture：收益407.17%，相对SOXX财富1.0171，MaxDD -44.21%，CVaR5 -5.46%，换手0.69。

开发期历史资格门通过：False。不晋升；所有历史已被查看，本次不能证明独立前向稳定超额。

线上反馈只使用决策前已经结算的专家净收益；组合有自己的持仓、费用与收益，不是专家净值的加权平均。固定混合用于隔离在线更新贡献，静态专家不参与事后选优。

年度差异见 annual_metrics.csv；2023以后仍为复用历史诊断。没有追加数据驱动的regime阈值，滚动窗口高度重叠，不算独立样本。

风险：回溯调整价格并非已独立核验的含息总回报；现金收益为零，历史开盘价不等于真实成交。基准核心与主动比例只是配置约束，不保证回撤或CVaR。

下一步：保留本轮失败或通过的全部证据；冻结后取得连续新时点记录。收益预测不确定性惩罚列为独立候选，须先有可信的过去OOS误差，不能根据本轮盈亏修改学习率或扩大候选。

Qlib Recorder：64c03a9f9c824e058a2bb360eb6aba11；as_of=2026-09-25。


# 基准相对架构与在线专家实验结论

运行了一个固定分享在线候选、固定混合对照及三个静态专家；全部独立扣费结算并通过原生 Qlib 回放。没有训练新预测网络或搜索参数。

每侧25bp，以下按信号年度切片，持仓与学习状态跨年连续：

- development / soxx：收益325.11%，相对SOXX财富1.0000，MaxDD -47.37%，CVaR5 -4.42%，换手1.00。
- reused_diagnostic / soxx：收益398.62%，相对SOXX财富1.0000，MaxDD -42.79%，CVaR5 -5.30%，换手0.00。
- development / leveraged：收益444.60%，相对SOXX财富1.2811，MaxDD -61.40%，CVaR5 -6.14%，换手7.95。
- reused_diagnostic / leveraged：收益607.41%，相对SOXX财富1.4187，MaxDD -56.90%，CVaR5 -7.34%，换手4.51。
- development / defensive：收益231.78%，相对SOXX财富0.7805，MaxDD -39.44%，CVaR5 -3.53%，换手2.53。
- reused_diagnostic / defensive：收益272.44%，相对SOXX财富0.7469，MaxDD -35.48%，CVaR5 -4.24%，换手1.10。
- development / online_fixed_share：收益330.09%，相对SOXX财富1.0117，MaxDD -48.76%，CVaR5 -4.57%，换手1.96。
- reused_diagnostic / online_fixed_share：收益407.71%，相对SOXX财富1.0182，MaxDD -44.30%，CVaR5 -5.49%，换手0.67。
- development / fixed_mixture：收益329.50%，相对SOXX财富1.0103，MaxDD -48.70%，CVaR5 -4.55%，换手2.02。
- reused_diagnostic / fixed_mixture：收益407.17%，相对SOXX财富1.0171，MaxDD -44.21%，CVaR5 -5.46%，换手0.69。

开发期历史资格门通过：False。不晋升；所有历史已被查看，本次不能证明独立前向稳定超额。

线上反馈只使用决策前已经结算的专家净收益；组合有自己的持仓、费用与收益，不是专家净值的加权平均。固定混合用于隔离在线更新贡献，静态专家不参与事后选优。

年度差异见 annual_metrics.csv；2023以后仍为复用历史诊断。没有追加数据驱动的regime阈值，滚动窗口高度重叠，不算独立样本。

风险：回溯调整价格并非已独立核验的含息总回报；现金收益为零，历史开盘价不等于真实成交。基准核心与主动比例只是配置约束，不保证回撤或CVaR。

下一步：保留本轮失败或通过的全部证据；冻结后取得连续新时点记录。收益预测不确定性惩罚列为独立候选，须先有可信的过去OOS误差，不能根据本轮盈亏修改学习率或扩大候选。

Qlib Recorder：64c03a9f9c824e058a2bb360eb6aba11；as_of=2026-09-25。

## 失败归因与下一阶段

25bp开发期7年中5年正超额，低于登记的6年要求；滚动252交易日跑赢比例57.97%，低于60%。候选MaxDD -48.76%、CVaR5 -4.57%，均差于同窗SOXX；删去最强年份后累计超额也不为正。两档成本、两种区块长度的置信下界均未高于零。

相对固定混合的25bp开发期终值改善仅0.137%，年化对数增量约1.95bp；已有SOXX核心配置解释了大部分表现，在线反馈没有提供足够增量证据。不能将略高终值包装为稳定alpha。

完整数据与代码身份在 run_binding.json；所有费用情景与失败门保留。下一机制需要可信的过去OOS预测误差来决定主动偏离惩罚，并取得冻结后的连续新证据；本次不根据失败结果改变专家、学习率或窗口。
